# Checkpoint 6: Sistemas recomendadores

## Estructura

Estrcutura de archivos:
```text
proyecto/
├── Checkpoint_6_Sistemas_Recomendadores.ipynb
└── data/
    ├── movies_metadata.csv
    ├── credits.csv
    └── keywords.csv
```


Las librerías necesarias son:

```text
pandas
numpy
scikit-learn
```

In [1]:
# instalar librerías
%pip install pandas numpy scikit-learn

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.5.1-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 1.5 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/9.9 MB 1.7 MB/s eta 0:00:06
   ----- ---------------------------------- 1.3/9.9 MB 1.9 MB/s eta 0:00:05
   ------- -------------------------------- 1.8/9.9 MB 2.0 MB/s eta 0:00:04
   -------- ------------------------------- 2.1/9.9 MB 2.0 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/9.9 MB 2.0 MB/s eta 0:00:04
   ------------ --------------------------- 3.1/9.9 MB 2.0 MB/s eta 0:00:04
   ------------- -------------------------- 3.4/9.9 MB 2.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
# Importación de librerías
from pathlib import Path
from ast import literal_eval

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

DATA_DIR = Path("./data")
MOVIES_FILE = DATA_DIR / "movies_metadata.csv"
KEYWORDS_FILE = DATA_DIR / "keywords.csv"
CREDITS_FILE = DATA_DIR / "credits.csv"

print("Carpeta de datos:", DATA_DIR.resolve())

Carpeta de datos: C:\Users\68323\Downloads\6_Recomendaciones\data


# 1. Carga y preparación de los datos

Primero se carga el archivo principal. También se convierten las columnas de votos a valores numéricos; esto evita errores cuando existen datos vacíos o mal formateados.

In [18]:
import pandas as pd
df = pd.read_csv(MOVIES_FILE, low_memory=False)
df.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afrai...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 10751, 'name': 'Family'}]",NaN,8844,tt0113497,en,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwitting...",17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'name': 'Teitler Film', 'id': 2550}, {'name': 'Interscope Communications'...","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'Français'}]",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collection', 'poster_path': '/nLvUdqgPgm3F85NMCii9gVFUcet.jpg', 'backdrop_pat...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'Comedy'}]",NaN,15602,tt0113228,en,Grumpier Old Men,"A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile,...",11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name': 'Lancaster Gate', 'id': 19464}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for Love.,Grumpier Old Men,False,6.5,92.0


In [26]:
# Revisar las columnas principales
columnas_interes = [
    "title",
    "overview",
    "vote_count",
    "vote_average",
    "genres",
    "release_date",
]

df[columnas_interes].head()

,title,overview,vote_count,vote_average,genres,release_date
0,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afrai...",5415.0,7.7,"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]",1995-10-30
1,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwitting...",2413.0,6.9,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 10751, 'name': 'Family'}]",1995-12-15
2,Grumpier Old Men,"A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile,...",92.0,6.5,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'Comedy'}]",1995-12-22
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the women are holding their breath, waiting for the elusive ""good man"" to bre...",34.0,6.1,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}]",1995-12-22
4,Father of the Bride Part II,"Just when George Banks has recovered from his daughter's wedding, he receives the news that she's pregnant ... and t...",173.0,5.7,"[{'id': 35, 'name': 'Comedy'}]",1995-02-10


In [20]:
#vote average = promedio de votos
C = df['vote_average'].mean()
print("Promedio de votos:", C)

Promedio de votos: 5.618207215134184


In [21]:
#numero de votos = cantidad de votos
m = df['vote_count'].quantile(0.90)
print("Número de votos:", m)

Número de votos: 160.0


# 2. Recomendador Inicial

Usar solamente la calificación promedio puede producir resultados engañosos. Una película con pocos votos podría tener una nota muy alta, aunque todavía no represente la opinión de muchas personas.

Por ello se utilizará una media ponderada que combina:

- `v`: número de votos de la película.
- `m`: mínimo de votos requerido.
- `R`: calificación promedio de la película.
- `C`: promedio general de todas las películas.

Fórmula en texto plano:

```text
score = (v / (v + m) * R) + (m / (v + m) * C)
```

In [22]:
#copia del dataframe original
q_movies = df.copy().loc[df['vote_count'] >= m]

In [23]:
#shape del dataframe
print("Dimensiones del conjunto de datos:", q_movies.shape)
print("Dimensiones del conjunto de datos original:", df.shape)

Dimensiones del conjunto de datos: (4555, 24)
Dimensiones del conjunto de datos original: (45466, 24)


In [ ]:
#Calcular calificacion ponderada de cada película
# media ponderada
def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']

    # Cálculo de IMDB
    return (v/(v+m) * R) + (m/(m+v) * C)

#score
q_movies['score'] = q_movies.apply(weighted_rating, axis=1)

#ordenar dataframe por score, mayor a menor
q_movies = q_movies.sort_values('score', ascending=False)

#Mostrar los primeros 15 resultados
q_movies[['title', 'vote_count', 'vote_average', 'score']].head(15)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


## Interpretación del recomendador simple

El sistema evita favorecer demasiado a películas con calificaciones altas pero con pocos votos. Una película obtiene una mejor posición cuando combina una buena calificación con una cantidad considerable de votos.

Su limitación es que muestra prácticamente las mismas recomendaciones a todas las personas; todavía no considera los gustos individuales.

# 3. Recomendador basado en contenido

Ahora se utilizará la columna `overview`, que contiene la descripción de la trama.

El procedimiento será:

1. Limpiar los valores vacíos.
2. Convertir las sinopsis en vectores TF-IDF.
3. Comparar las películas mediante similitud coseno.
4. Devolver las películas con mayor similitud.

Esta versión calcula únicamente la fila de similitudes necesaria en cada consulta; así evita crear una matriz completa de 45,000 por 45,000 elementos y utiliza menos memoria.

In [ ]:
#inspeccionar overview
df['overview'].head()

0    Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afrai...
1    When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwitting...
2    A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile,...
3    Cheated on, mistreated and stepped on, the women are holding their breath, waiting for the elusive "good man" to bre...
4    Just when George Banks has recovered from his daughter's wedding, he receives the news that she's pregnant ... and t...
Name: overview, dtype: str

# Vectorización TF-IDF

In [30]:
#Importar TfIdfVectorizer de scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer

#Definir el objeto de la clase TF-IDF Vectorizer. Quitamos stop words de inglés
tfidf = TfidfVectorizer(stop_words='english')

#Reemplazar NaN por string vacío
df['overview'] = df['overview'].fillna('')

# Consruir la matriz TF-IDF haciendo ajustes y transformaciones
tfidf_matrix = tfidf.fit_transform(df['overview'])

#Mostrar shape
tfidf_matrix.shape

(45466, 75827)

# Opcional, remover palabras que aparecen en menos de 2 documentos

In [29]:
#Importar TfIdfVectorizer de scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer

# Definir el objeto de la clase TF-IDF Vectorizer. Quitamos stop words de inglés
tfidf = TfidfVectorizer(
    stop_words="english",
    min_df=2
)

# Reemplazar valores vacíos por texto vacío
df["overview"] = df["overview"].fillna("")

# Construir la matriz TF-IDF
tfidf_matrix = tfidf.fit_transform(df["overview"])

#Mostrar shape
print("Forma de la matriz TF-IDF:", tfidf_matrix.shape)
print("Número de palabras diferentes:", len(tfidf.get_feature_names_out()))

Forma de la matriz TF-IDF: (45466, 37664)
Número de palabras diferentes: 37664


df significa Document Frequency.

Es decir:
¿En cuántos documentos aparece una palabra?

No importa cuántas veces aparezca dentro del mismo documento.

En resumen, min_df es un filtro que elimina palabras demasiado raras. Cuanto mayor sea su valor, más pequeño será el vocabulario y más ligera será la matriz TF-IDF

Se reduce a un poco mas de la mitad el total de palabras diferentes. 
Usar min_df=2, 3 o incluso 5 suele ser una buena idea: Reduce el ruido, disminuye el uso de memoria y acelera los cálculos, generalmente sin afectar mucho la calidad de las recomendaciones.

In [34]:
# Mostrar algunas palabras del vocabulario
tfidf.get_feature_names_out()[5000:5010]

array(['avails', 'avaks', 'avalanche', 'avalanches', 'avallone', 'avalon',
       'avant', 'avanthika', 'avanti', 'avaracious'], dtype=object)

Puntuación de similtud

cos(x,y) = \frac{\bf x \cdot \bf y^{\intercal}}{\lvert\lvert \bf  x \rvert\rvert \cdot \lvert\lvert \bf y \rvert\rvert} = \frac{\sum^{n}_{i=1}\bf x_i \cdot \bf y_{i}^\intercal}{\sqrt{\sum^{n}_{i=1}(\bf x_i)^2}\sqrt{\sum^{n}_{i=1}(\bf y_i)^2}}

In [35]:
from sklearn.metrics.pairwise import linear_kernel

# Calcular la matriz de similitud coseno
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

#forma
cosine_sim.shape

(45466, 45466)

Mapeo inverso

In [36]:
#Construct a reverse map of indices and movie titles
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
indices[:10]

title
Toy Story                      0
Jumanji                        1
Grumpier Old Men               2
Waiting to Exhale              3
Father of the Bride Part II    4
Heat                           5
Sabrina                        6
Tom and Huck                   7
Sudden Death                   8
GoldenEye                      9
dtype: int64

Función de recomendación

In [37]:
# Función que toma el título de una película como entrada y devuelve las películas más similares
def get_recommendations(title, cosine_sim=cosine_sim):
    # Obtener el índice de la película que coincide con el título
    idx = indices[title]

    # Obtener las puntuaciones de similitud por pares de todas las películas con esa película
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Ordenar las películas según las puntuaciones de similitud
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Obtener las puntuaciones de las 10 películas más similares
    sim_scores = sim_scores[1:11]

    # Obtener los índices de las películas
    movie_indices = [i[0] for i in sim_scores]

    # Devolver las 10 películas más similares
    return df['title'].iloc[movie_indices]

In [38]:
#usando funcion get_recommendations
get_recommendations('The Dark Knight Rises')

12481                                       The Dark Knight
150                                          Batman Forever
1328                                         Batman Returns
15511                            Batman: Under the Red Hood
585                                                  Batman
21194    Batman Unmasked: The Psychology of the Dark Knight
9230                     Batman Beyond: Return of the Joker
18035                                      Batman: Year One
19792               Batman: The Dark Knight Returns, Part 1
3095                           Batman: Mask of the Phantasm
Name: title, dtype: str

In [39]:
#usando funcion get_recommendations
get_recommendations('The Godfather')

1178               The Godfather: Part II
44030    The Godfather Trilogy: 1972-1990
1914              The Godfather: Part III
23126                          Blood Ties
11297                    Household Saints
34717                   Start Liquidation
10821                            Election
38030            A Mother Should Be Loved
17729                   Short Sharp Shock
26293                  Beck 28 - Familjen
Name: title, dtype: str

# 3. Mejorando el recomendador

In [51]:
# cargar conjuntos de datos adicionales
credits = pd.read_csv('./data/credits.csv')
keywords = pd.read_csv('./data/keywords.csv')

# Eliminar algunos IDs problemáticos
df = df.drop([19730, 29503, 35587, 35803])

# Convetir todos los ids a números enteros
keywords['id'] = keywords['id'].astype('int')
credits['id'] = credits['id'].astype('int')
df['id'] = df['id'].astype('int')

# Hacer merges entre data frames
df = df.merge(credits, on='id')
df = df.merge(keywords, on='id')

In [52]:
#exploracion de df
# Parse the stringified features into their corresponding python objects
from ast import literal_eval

features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(literal_eval)

In [53]:
#extraccion de datos
import numpy as np
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [54]:
#nueva funcion
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        #Checar si existen más de 3 elementos. Si sí, regresar primeros 3, si no, todos
        if len(names) > 3:
            names = names[:3]
        return names

    # regresar lista vacía si los datos no están bien formateados
    return []

In [55]:
#ejemplos
# Extraer director de columnas crew
df['director'] = df['crew'].apply(get_director)

# Extraer top 3 de elenco, palabras clave y géneros
features = ['cast', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(get_list)

In [57]:
#continuar procesamiento
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        #Checar si existse el director. Si no, regresar ""
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''
        
features = ['cast', 'keywords', 'director', 'genres']

for feature in features:
    df[feature] = df[feature].apply(clean_data)


In [58]:
#metadata soup
def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])

df['soup'] = df.apply(create_soup, axis=1)

df[['soup']].head(2)

,soup
0,jealousy toy boy tomhanks timallen donrickles johnlasseter animation comedy family
1,boardgame disappearance basedonchildren'sbook robinwilliams jonathanhyde kirstendunst joejohnston adventure fantasy ...


In [59]:
#Vectorizador CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df['soup'])


count_matrix.shape

(46626, 73880)

In [60]:
#similitud coseno
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim2 = cosine_similarity(count_matrix, count_matrix)

# Reset index of your main DataFrame and construct reverse mapping as before
df = df.reset_index()
indices = pd.Series(df.index, index=df['title'])

In [61]:
#reutilizacion de funcion get_recommendations
get_recommendations('The Dark Knight Rises', cosine_sim2)

12541      The Dark Knight
10170        Batman Begins
9271                Shiner
9834       Amongst Friends
7732              Mitchell
516      Romeo Is Bleeding
11411         The Prestige
24040            Quicksand
24984             Deadfall
41041                 Sara
Name: title, dtype: str

In [63]:
get_recommendations('The Godfather', cosine_sim2)

1926            The Godfather: Part III
1187             The Godfather: Part II
15534                   The Rain People
18866                         Last Exit
34458                              Rege
35772            Manuscripts Don't Burn
35773            Manuscripts Don't Burn
7961     The Night of the Following Day
18187                 The Son of No One
28637            In the Name of the Law
Name: title, dtype: str

## Interpretación del recomendador por sinopsis

Las recomendaciones suelen compartir palabras o temas de la trama. Por ejemplo, una película de Batman puede devolver otras películas relacionadas con el mismo personaje.

Sin embargo, este método todavía desconoce información importante como el director, el reparto y los géneros. Por esa razón algunas sugerencias pueden parecer relacionadas por texto, pero no necesariamente por estilo cinematográfico.

# 4. Recomendador mejorado con metadatos

En esta parte se agregan:

- Los tres actores principales.
- El director.
- Los tres géneros principales.
- Las tres palabras clave principales.

Con estos datos se construirá una cadena llamada `soup`, que resume los metadatos más importantes de cada película.

In [81]:
# Comprobar que existan los archivos adicionales
archivos_faltantes = [
    str(ruta)
    for ruta in [CREDITS_FILE, CREDITS_FILE, KEYWORDS_FILE]
    if not ruta.exists()
]

if archivos_faltantes:
    raise FileNotFoundError(
        "Faltan los siguientes archivos:\n- "
        + "\n- ".join(archivos_faltantes)
    )

credits = pd.read_csv(CREDITS_FILE)
keywords = pd.read_csv(KEYWORDS_FILE)

print("Credits:", credits.shape)
print("Keywords:", keywords.shape)

Credits: (45476, 3)
Keywords: (46419, 2)


In [82]:
# Preparar identificadores antes de combinar los archivos
movies_meta = df.copy()

movies_meta["id"] = pd.to_numeric(movies_meta["id"], errors="coerce")
credits["id"] = pd.to_numeric(credits["id"], errors="coerce")
keywords["id"] = pd.to_numeric(keywords["id"], errors="coerce")

# Eliminar registros sin ID válido
movies_meta = movies_meta.dropna(subset=["id"]).copy()
credits = credits.dropna(subset=["id"]).copy()
keywords = keywords.dropna(subset=["id"]).copy()

# Convertir ID a entero
movies_meta["id"] = movies_meta["id"].astype(int)
credits["id"] = credits["id"].astype(int)
keywords["id"] = keywords["id"].astype(int)

# Eliminar duplicados antes de combinar
credits = credits.drop_duplicates(subset="id")
keywords = keywords.drop_duplicates(subset="id")

# Combinar los DataFrames
movies_meta = movies_meta.merge(credits, on="id", how="inner")
movies_meta = movies_meta.merge(keywords, on="id", how="inner")

print("Dimensiones después de los merges:", movies_meta.shape)
movies_meta.head(2)

Dimensiones después de los merges: (46626, 33)


,index,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,cast_x,crew_x,keywords_x,director,soup,cast_y,crew_y,keywords_y
0,0,False,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/...",30000000,"[animation, comedy, family]",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afrai...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,"[tomhanks, timallen, donrickles]","[{'credit_id': '52fe4284c3a36847f8024f49', 'department': 'Directing', 'gender': 2, 'id': 7879, 'job': 'Director', 'n...","[jealousy, toy, boy]",johnlasseter,jealousy toy boy tomhanks timallen donrickles johnlasseter animation comedy family,"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name...","[{'credit_id': '52fe4284c3a36847f8024f49', 'department': 'Directing', 'gender': 2, 'id': 7879, 'job': 'Director', 'n...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'fr..."
1,1,False,NaN,65000000,"[adventure, fantasy, family]",NaN,8844,tt0113497,en,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwitting...",17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'name': 'Teitler Film', 'id': 2550}, {'name': 'Interscope Communications'...","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'Français'}]",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,"[robinwilliams, jonathanhyde, kirstendunst]","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'department': 'Production', 'gender': 2, 'id': 511, 'job': 'Executive Pro...","[boardgame, disappearance, basedonchildren'sbook]",joejohnston,boardgame disappearance basedonchildren'sbook robinwilliams jonathanhyde kirstendunst joejohnston adventure fantasy ...,"[{'cast_id': 1, 'character': 'Alan Parrish', 'credit_id': '52fe44bfc3a36847f80a7c73', 'gender': 2, 'id': 2157, 'name...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'department': 'Production', 'gender': 2, 'id': 511, 'job': 'Executive Pro...","[{'id': 10090, 'name': 'board game'}, {'id': 10941, 'name': 'disappearance'}, {'id': 15101, 'name': ""based on childr..."


In [83]:
# Volver a cargar los datos originales para evitar columnas duplicadas
movies_meta = pd.read_csv(MOVIES_FILE, low_memory=False)

credits = pd.read_csv(CREDITS_FILE)
keywords = pd.read_csv(KEYWORDS_FILE)

# Convertir los identificadores a valores numéricos
movies_meta["id"] = pd.to_numeric(
    movies_meta["id"],
    errors="coerce"
)

credits["id"] = pd.to_numeric(
    credits["id"],
    errors="coerce"
)

keywords["id"] = pd.to_numeric(
    keywords["id"],
    errors="coerce"
)

# Eliminar registros con identificadores inválidos
movies_meta = movies_meta.dropna(
    subset=["id"]
).copy()

credits = credits.dropna(
    subset=["id"]
).copy()

keywords = keywords.dropna(
    subset=["id"]
).copy()

# Convertir los identificadores a enteros
movies_meta["id"] = movies_meta["id"].astype(int)
credits["id"] = credits["id"].astype(int)
keywords["id"] = keywords["id"].astype(int)

# Eliminar registros duplicados
movies_meta = movies_meta.drop_duplicates(
    subset="id"
)

credits = credits.drop_duplicates(
    subset="id"
)

keywords = keywords.drop_duplicates(
    subset="id"
)

# Combinar los tres archivos
movies_meta = movies_meta.merge(
    credits,
    on="id",
    how="inner"
)

movies_meta = movies_meta.merge(
    keywords,
    on="id",
    how="inner"
)

print(
    "Dimensiones después de los merges:",
    movies_meta.shape
)

print(
    "Columnas disponibles:",
    movies_meta.columns.tolist()
)

movies_meta.head(2)

Dimensiones después de los merges: (45432, 27)
Columnas disponibles: ['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'cast', 'crew', 'keywords']


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,cast,crew,keywords
0,False,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afrai...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name...","[{'credit_id': '52fe4284c3a36847f8024f49', 'department': 'Directing', 'gender': 2, 'id': 7879, 'job': 'Director', 'n...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'fr..."
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 10751, 'name': 'Family'}]",NaN,8844,tt0113497,en,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwitting...",17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'name': 'Teitler Film', 'id': 2550}, {'name': 'Interscope Communications'...","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'Français'}]",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,"[{'cast_id': 1, 'character': 'Alan Parrish', 'credit_id': '52fe44bfc3a36847f80a7c73', 'gender': 2, 'id': 2157, 'name...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'department': 'Production', 'gender': 2, 'id': 511, 'job': 'Executive Pro...","[{'id': 10090, 'name': 'board game'}, {'id': 10941, 'name': 'disappearance'}, {'id': 15101, 'name': ""based on childr..."


In [ ]:
#CONVERSOR DE COLUMNAS JSON
def parse_list(value):
    """Convierte una lista guardada como texto a una lista de Python."""

    if isinstance(value, list):
        return value

    if not isinstance(value, str) or value.strip() == "":
        return []

    try:
        parsed = literal_eval(value)

        if isinstance(parsed, list):
            return parsed

        return []

    except (ValueError, SyntaxError):
        return []

# Convertir las columnas serializadas
for feature in ["cast", "crew", "keywords", "genres"]:
    movies_meta[feature] = movies_meta[feature].apply(
        parse_list
    )

In [ ]:
#EXTRACCION DE DIRECTOR, REPARTO Y GENEROS
def get_director(crew):
    """Devuelve el nombre del director o una cadena vacía."""
    for person in crew:
        if isinstance(person, dict) and person.get("job") == "Director":
            return person.get("name", "")
    return ""

print(movies_meta.loc[0, "cast"][:3])
print(movies_meta.loc[0, "keywords"][:3])
print(movies_meta.loc[0, "genres"][:3])

def get_top_names(items, limit=3):
    """Obtiene los primeros nombres sin vaciar datos ya procesados."""
    if not isinstance(items, list):
        return []

    names = []

    for item in items:
        if isinstance(item, dict):
            name = item.get("name", "")

            if name:
                names.append(name)

        elif isinstance(item, str) and item.strip():
            names.append(item.strip())

    return names[:limit]


movies_meta["director"] = movies_meta["crew"].apply(get_director)

for feature in ["cast", "keywords", "genres"]:
    movies_meta[feature] = movies_meta[feature].apply(get_top_names)

movies_meta[
    ["title", "cast", "director", "keywords", "genres"]
].head(3)

[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}]
[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}]
[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]


,title,cast,director,keywords,genres
0,Toy Story,"[Tom Hanks, Tim Allen, Don Rickles]",John Lasseter,"[jealousy, toy, boy]","[Animation, Comedy, Family]"
1,Jumanji,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]",Joe Johnston,"[board game, disappearance, based on children's book]","[Adventure, Fantasy, Family]"
2,Grumpier Old Men,"[Walter Matthau, Jack Lemmon, Ann-Margret]",Howard Deutch,"[fishing, best friend, duringcreditsstinger]","[Romance, Comedy]"


In [86]:
def clean_data(value):
    """Convierte texto a minúsculas y elimina espacios internos."""
    if isinstance(value, list):
        return [
            str(item).lower().replace(" ", "")
            for item in value
            if str(item).strip()
        ]

    if isinstance(value, str):
        return value.lower().replace(" ", "")

    return ""


for feature in ["cast", "keywords", "director", "genres"]:
    movies_meta[feature] = movies_meta[feature].apply(clean_data)

In [87]:
def create_soup(movie):
    """Une palabras clave, reparto, director y géneros."""
    return " ".join(
        movie["keywords"]
        + movie["cast"]
        + [movie["director"], movie["director"]]  # Se repite para dar mayor peso
        + movie["genres"]
    )


movies_meta["soup"] = movies_meta.apply(create_soup, axis=1)

movies_meta[["title", "soup"]].head(3)

,title,soup
0,Toy Story,jealousy toy boy tomhanks timallen donrickles johnlasseter johnlasseter animation comedy family
1,Jumanji,boardgame disappearance basedonchildren'sbook robinwilliams jonathanhyde kirstendunst joejohnston joejohnston advent...
2,Grumpier Old Men,fishing bestfriend duringcreditsstinger waltermatthau jacklemmon ann-margret howarddeutch howarddeutch romance comedy


Se repite dos veces el director dentro de `soup` para darle un poco más de importancia. Esto puede mejorar las recomendaciones cuando varias películas comparten al mismo director.

In [88]:
# Vectorizar los metadatos
count = CountVectorizer(stop_words="english")
count_matrix = count.fit_transform(movies_meta["soup"])

print("Forma de la matriz de metadatos:", count_matrix.shape)
print("Número de términos:", len(count.get_feature_names_out()))

Forma de la matriz de metadatos: (45432, 73880)
Número de términos: 73880


In [89]:
# Reiniciar índices para que coincidan con la matriz
movies_meta = movies_meta.reset_index(drop=True)

indices_meta = (
    pd.Series(movies_meta.index, index=movies_meta["title"])
    .loc[lambda s: ~s.index.duplicated(keep="first")]
)


def get_recommendations_metadata(title, n=10):
    """Recomienda películas usando actores, director, géneros y palabras clave."""
    if title not in indices_meta:
        coincidencias = [
            t for t in indices_meta.index
            if isinstance(t, str) and title.lower() in t.lower()
        ][:10]
        raise KeyError(
            f"No se encontró el título '{title}'. "
            f"Posibles coincidencias: {coincidencias}"
        )

    idx = int(indices_meta[title])

    # Comparar una película con todas las demás
    sim_scores = cosine_similarity(
        count_matrix[idx:idx + 1],
        count_matrix
    ).flatten()

    orden = sim_scores.argsort()[::-1]
    orden = [i for i in orden if i != idx][:n]

    resultado = movies_meta.loc[
        orden,
        ["title", "director", "cast", "genres", "vote_average", "vote_count"]
    ].copy()

    resultado["similarity"] = sim_scores[orden]

    return resultado.reset_index(drop=True)

EJEMPLOS:

In [90]:
# Recomendaciones mejoradas para The Dark Knight Rises
get_recommendations_metadata("The Dark Knight Rises")

,title,director,cast,genres,vote_average,vote_count,similarity
0,The Dark Knight,christophernolan,"[christianbale, michaelcaine, heathledger]","[drama, action, crime]",8.3,12269.0,0.846154
1,Batman Begins,christophernolan,"[christianbale, michaelcaine, liamneeson]","[action, crime, drama]",7.5,7511.0,0.769231
2,The Prestige,christophernolan,"[hughjackman, christianbale, michaelcaine]","[drama, mystery, thriller]",8.0,4510.0,0.538462
3,Following,christophernolan,"[jeremytheobald, alexhaw, lucyrussell]","[crime, drama, thriller]",7.2,363.0,0.461538
4,Dunkirk,christophernolan,"[fionnwhitehead, tomglynn-carney, jacklowden]","[action, drama, history]",7.5,2712.0,0.444750
5,Interstellar,christophernolan,"[matthewmcconaughey, jessicachastain, annehathaway]","[adventure, drama, sciencefiction]",8.1,11187.0,0.384615
6,Insomnia,christophernolan,"[alpacino, robinwilliams, hilaryswank]","[crime, mystery, thriller]",6.8,1181.0,0.384615
7,Inception,christophernolan,"[leonardodicaprio, josephgordon-levitt, ellenpage]","[action, thriller, sciencefiction]",8.1,14075.0,0.370625
8,Doodlebug,christophernolan,[jeremytheobald],"[fantasy, mystery]",6.6,105.0,0.369800
9,Shiner,johnirvin,"[michaelcaine, martinlandau, andyserkis]","[drama, action, crime]",5.1,10.0,0.350823


In [91]:
# Recomendaciones mejoradas para The Godfather
get_recommendations_metadata("The Godfather")

,title,director,cast,genres,vote_average,vote_count,similarity
0,The Godfather: Part III,francisfordcoppola,"[alpacino, dianekeaton, andygarcía]","[crime, drama, thriller]",7.1,1589.0,0.640513
1,The Rain People,francisfordcoppola,"[jamescaan, shirleyknight, robertduvall]",[drama],5.5,10.0,0.577350
2,The Godfather: Part II,francisfordcoppola,"[alpacino, robertduvall, dianekeaton]","[drama, crime]",8.3,3418.0,0.560449
3,Tucker: The Man and His Dream,francisfordcoppola,"[jeffbridges, joanallen, martinlandau]",[drama],6.5,71.0,0.510310
4,Gardens of Stone,francisfordcoppola,"[jamescaan, anjelicahuston, jamesearljones]","[drama, history]",5.5,25.0,0.500000
5,Apocalypse Now,francisfordcoppola,"[martinsheen, marlonbrando, robertduvall]","[drama, war]",8.0,2112.0,0.500000
6,The Outsiders,francisfordcoppola,"[mattdillon, ralphmacchio, c.thomashowell]","[crime, drama]",6.9,293.0,0.480384
7,The Conversation,francisfordcoppola,"[genehackman, johncazale, fredericforrest]","[crime, drama, mystery]",7.5,377.0,0.480384
8,The Cotton Club,francisfordcoppola,"[richardgere, gregoryhines, dianelane]","[music, drama, crime]",6.5,71.0,0.480384
9,The Rainmaker,francisfordcoppola,"[mattdamon, dannydevito, jonvoight]","[drama, crime, thriller]",6.7,239.0,0.480384


# SUGERENCIAS

***

## 1. Recomendador híbrido: similitud y popularidad (INTRODUCIR FILTRO DE POPULARIDAD)

Una mejora consiste en:

1. Buscar primero las 30 películas más similares.
2. Calcular su calificación ponderada.
3. Mostrar las 10 mejores dentro de ese grupo.

Así se evita recomendar una película muy similar pero poco valorada por los usuarios.

In [92]:
def get_hybrid_recommendations(title, candidates=30, n=10):
    """Combina similitud de contenido con la calificación ponderada."""
    if title not in indices_meta:
        raise KeyError(f"No se encontró el título '{title}'.")

    idx = int(indices_meta[title])

    sim_scores = cosine_similarity(
        count_matrix[idx:idx + 1],
        count_matrix
    ).flatten()

    orden = sim_scores.argsort()[::-1]
    orden = [i for i in orden if i != idx][:candidates]

    resultado = movies_meta.loc[
        orden,
        ["title", "director", "genres", "vote_average", "vote_count"]
    ].copy()

    resultado["similarity"] = sim_scores[orden]

    # Convertir a numérico por seguridad
    resultado["vote_average"] = pd.to_numeric(
        resultado["vote_average"],
        errors="coerce"
    )
    resultado["vote_count"] = pd.to_numeric(
        resultado["vote_count"],
        errors="coerce"
    )

    resultado = resultado.dropna(
        subset=["vote_average", "vote_count"]
    ).copy()

    resultado["weighted_score"] = resultado.apply(
        weighted_rating,
        axis=1
    )

    return (
        resultado.sort_values(
            ["weighted_score", "similarity"],
            ascending=False
        )
        .head(n)
        .reset_index(drop=True)
    )

EJEMPLO

In [93]:
# Ejemplo del recomendador híbrido
get_hybrid_recommendations("The Dark Knight Rises")

,title,director,genres,vote_average,vote_count,similarity,weighted_score
0,The Dark Knight,christophernolan,"[drama, action, crime]",8.3,12269.0,0.846154,8.265477
1,Inception,christophernolan,"[action, thriller, sciencefiction]",8.1,14075.0,0.370625,8.072105
2,Interstellar,christophernolan,"[adventure, drama, sciencefiction]",8.1,11187.0,0.384615,8.065005
3,Memento,christophernolan,"[mystery, thriller]",8.1,4168.0,0.307692,8.008252
4,The Prestige,christophernolan,"[drama, mystery, thriller]",8.0,4510.0,0.538462,7.918397
5,Batman Begins,christophernolan,"[action, crime, drama]",7.5,7511.0,0.769231,7.460750
6,Dunkirk,christophernolan,"[action, drama, history]",7.5,2712.0,0.444750,7.395165
7,Following,christophernolan,"[crime, drama, thriller]",7.2,363.0,0.461538,6.716086
8,Insomnia,christophernolan,"[crime, mystery, thriller]",6.8,1181.0,0.384615,6.658996
9,Doodlebug,christophernolan,"[fantasy, mystery]",6.6,105.0,0.369800,6.007219


## 2. Incluir otros miembros del equipo (GUINISTAS O PRODUCTORES)

En la versión anterior del recomendador basado en contenido únicamente se consideraban el director, los actores principales, los géneros y las palabras clave para representar cada película.

Como mejora, se incorporan otros miembros relevantes del equipo de producción, específicamente guionistas y productores. Estos profesionales también influyen en el estilo, la narrativa y la calidad de una película, por lo que incluirlos permite construir una representación más completa del contenido.

Esta información se añadirá al campo `soup`, incrementando la cantidad de características utilizadas para calcular la similitud entre películas. De esta manera, el sistema podrá identificar relaciones entre películas que comparten talento creativo detrás de cámaras, mejorando potencialmente la calidad de las recomendaciones.

In [94]:
def get_crew_names(crew, jobs, limit=3):
    """Obtiene nombres de miembros del equipo según su puesto."""
    if not isinstance(crew, list):
        return []

    names = []

    for person in crew:
        if not isinstance(person, dict):
            continue

        job = person.get("job", "")
        name = person.get("name", "")

        if job in jobs and name:
            names.append(name)

    # Evitar nombres duplicados
    names = list(dict.fromkeys(names))

    return names[:limit]


movies_meta["writers"] = movies_meta["crew"].apply(
    lambda crew: get_crew_names(
        crew,
        jobs=[
            "Writer",
            "Screenplay",
            "Story",
            "Novel",
            "Characters"
        ],
        limit=3
    )
)

movies_meta["producers"] = movies_meta["crew"].apply(
    lambda crew: get_crew_names(
        crew,
        jobs=[
            "Producer",
            "Executive Producer",
            "Co-Producer",
            "Associate Producer"
        ],
        limit=3
    )
)

movies_meta[
    [
        "title",
        "director",
        "writers",
        "producers"
    ]
].head(3)

def create_soup(row):
    cast = " ".join(row["cast"])
    keywords = " ".join(row["keywords"])
    genres = " ".join(row["genres"])
    director = row["director"]
    writers = " ".join(row["writers"])
    producers = " ".join(row["producers"])

    return (
        f"{cast} "
        f"{keywords} "
        f"{genres} "
        f"{director} {director} {director} "
        f"{writers} "
        f"{producers}"
    )

movies_meta["soup"] = movies_meta.apply(
    create_soup,
    axis=1
)


EJEMPLO

In [101]:
# Ejemplo del recomendador miembros del equipo
titulo = "The Dark Knight Rises"

pelicula = movies_meta.loc[
    movies_meta["title"] == titulo
]

if pelicula.empty:
    print(f"No se encontró la película: {titulo}")
else:
    print("Título:", titulo)
    print("Director:", pelicula["director"].iloc[0])
    print("Guionistas:", pelicula["writers"].iloc[0])
    print("Productores:", pelicula["producers"].iloc[0])

Título: The Dark Knight Rises
Director: christophernolan
Guionistas: ['Christopher Nolan', 'Jonathan Nolan', 'David S. Goyer']
Productores: ['Charles Roven', 'Christopher Nolan', 'Emma Thomas']


## 3. Aumentar el peso del director

En un sistema de recomendación basado en contenido, no todas las características de una película tienen la misma importancia al momento de determinar su similitud con otras. En muchos casos, el director representa un elemento distintivo, ya que suele imprimir un estilo narrativo, visual y creativo que se mantiene a lo largo de su filmografía.

Como mejora al modelo, se incrementará la relevancia del director dentro de la representación textual (`soup`) utilizada para calcular la similitud entre películas. Para ello, el nombre del director se repetirá varias veces dentro de dicha representación, incrementando su frecuencia y, por consecuencia, su peso durante el proceso de vectorización mediante `CountVectorizer`.

Esta estrategia no modifica el algoritmo de similitud, sino que ajusta la representación de los datos de entrada para favorecer que películas dirigidas por la misma persona obtengan puntuaciones de similitud más altas. De esta manera, el recomendador puede capturar con mayor precisión el estilo cinematográfico característico de determinados directores y generar recomendaciones más coherentes desde una perspectiva artística.

In [102]:
def clean_feature(value):
    """
    Normaliza nombres para que CountVectorizer los interprete
    como una sola característica.

    Ejemplo:
    'Christopher Nolan' -> 'christophernolan'
    """
    if not isinstance(value, str):
        return ""

    return value.lower().replace(" ", "")


def clean_feature_list(values):
    """Normaliza los elementos contenidos en una lista."""
    if not isinstance(values, list):
        return []

    return [
        clean_feature(value)
        for value in values
        if isinstance(value, str) and value.strip()
    ]


# Normalizar las características
for feature in [
    "cast",
    "keywords",
    "genres",
    "writers",
    "producers"
]:
    movies_meta[feature] = movies_meta[feature].apply(
        clean_feature_list
    )

movies_meta["director"] = movies_meta["director"].apply(
    clean_feature
)

In [ ]:
#soup final
#director_weight = 3, el director aparece 3 veces en la sopa para aumentar su importancia
def create_weighted_soup(row, director_weight=3):
    """
    Construye la representación textual de cada película.

    El director se repite varias veces para incrementar
    su importancia en el cálculo de similitud.
    """
    cast = " ".join(row["cast"])
    keywords = " ".join(row["keywords"])
    genres = " ".join(row["genres"])
    writers = " ".join(row["writers"])
    producers = " ".join(row["producers"])

    director = row["director"]

    # Repetir el director para aumentar su peso
    weighted_director = " ".join(
        [director] * director_weight
    )

    return " ".join(
        [
            cast,
            keywords,
            genres,
            weighted_director,
            writers,
            producers
        ]
    ).strip()


movies_meta["soup"] = movies_meta.apply(
    create_weighted_soup,
    axis=1
)

EJEMPLO

In [104]:
movies_meta.loc[
    movies_meta["title"] == "The Dark Knight Rises",
    [
        "title",
        "director",
        "cast",
        "writers",
        "producers",
        "soup"
    ]
]

,title,director,cast,writers,producers,soup
18244,The Dark Knight Rises,christophernolan,"[christianbale, michaelcaine, garyoldman]","[christophernolan, jonathannolan, davids.goyer]","[charlesroven, christophernolan, emmathomas]",christianbale michaelcaine garyoldman dccomics crimefighter terrorist action crime drama christophernolan christophe...


In [ ]:
# BORRAR MATRIZ AND SIMILITUD PARA LIBERAR MEMORIA
del count_matrix
del cosine_sim

import gc
gc.collect()



NameError: name 'count_matrix' is not defined

In [113]:
movies_demo = movies_meta.sample(
    5000,
    random_state=42
).copy()


count_matrix = count_vectorizer.fit_transform(
    movies_demo["soup"]
)

cosine_sim = cosine_similarity(count_matrix)

In [114]:
# Película utilizada como ejemplo
titulo_ejemplo = "The Dark Knight Rises"

# Obtener la fila de la película consultada
pelicula_objetivo = movies_meta[
    movies_meta["title"] == titulo_ejemplo
].copy()

if pelicula_objetivo.empty:
    raise ValueError(
        f"No se encontró la película: {titulo_ejemplo}"
    )

# Tomar 4,999 películas distintas de la película objetivo
otras_peliculas = movies_meta[
    movies_meta["title"] != titulo_ejemplo
].sample(
    4999,
    random_state=42
)

# Crear una muestra que incluya obligatoriamente la película objetivo
movies_demo = pd.concat(
    [pelicula_objetivo, otras_peliculas],
    ignore_index=True
)

print("Películas utilizadas:", len(movies_demo))
print(
    "Película objetivo incluida:",
    titulo_ejemplo in movies_demo["title"].values
)

def create_soup_with_director_weight(row, director_weight=1):
    """
    Construye la sopa de características y permite controlar
    el peso asignado al director.
    """
    cast = " ".join(row["cast"])
    keywords = " ".join(row["keywords"])
    genres = " ".join(row["genres"])

    writers = " ".join(
        row["writers"]
        if isinstance(row["writers"], list)
        else []
    )

    producers = " ".join(
        row["producers"]
        if isinstance(row["producers"], list)
        else []
    )

    director = row["director"]

    # Repetir el director según el peso indicado
    weighted_director = " ".join(
        [director] * director_weight
    )

    return " ".join(
        [
            cast,
            keywords,
            genres,
            weighted_director,
            writers,
            producers
        ]
    ).strip()


# Sopa con peso normal del director
movies_demo["soup_director_1"] = movies_demo.apply(
    lambda row: create_soup_with_director_weight(
        row,
        director_weight=1
    ),
    axis=1
)

# Sopa con peso aumentado del director
movies_demo["soup_director_3"] = movies_demo.apply(
    lambda row: create_soup_with_director_weight(
        row,
        director_weight=3
    ),
    axis=1
)

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def recommend_with_director_weight(
    title,
    dataframe,
    soup_column,
    n=10
):
    """
    Recomienda películas utilizando la columna de sopa indicada.
    Calcula solamente la similitud de la película consultada
    contra las demás películas.
    """
    coincidencias = dataframe.index[
        dataframe["title"] == title
    ].tolist()

    if not coincidencias:
        raise ValueError(
            f"No se encontró la película: {title}"
        )

    movie_index = coincidencias[0]

    vectorizer = CountVectorizer(
        stop_words="english"
    )

    feature_matrix = vectorizer.fit_transform(
        dataframe[soup_column].fillna("")
    )

    # Comparar una película contra todas las demás
    similarity_scores = cosine_similarity(
        feature_matrix[movie_index],
        feature_matrix
    ).flatten()

    # Ordenar de mayor a menor similitud
    similar_indices = similarity_scores.argsort()[::-1]

    # Excluir la propia película
    similar_indices = [
        index
        for index in similar_indices
        if index != movie_index
    ][:n]

    result = dataframe.loc[
        similar_indices,
        [
            "title",
            "director",
            "genres",
            "vote_average"
        ]
    ].copy()

    result["similarity"] = similarity_scores[
        similar_indices
    ]

    result["same_director"] = (
        result["director"]
        == dataframe.loc[movie_index, "director"]
    )

    return result.reset_index(drop=True)




Películas utilizadas: 5000
Película objetivo incluida: True


EJEMPLO

In [116]:
recomendaciones_peso_1 = recommend_with_director_weight(
    title="The Dark Knight Rises",
    dataframe=movies_demo,
    soup_column="soup_director_1",
    n=10
)

print("Recomendaciones con peso normal del director:")

recomendaciones_peso_1


Recomendaciones con peso normal del director:


,title,director,genres,vote_average,similarity,same_director
0,Nick Fury: Agent of S.H.I.E.L.D.,rodhardy,"[sciencefiction, action]",3.4,0.251478,False
1,Gangsters,giuseppevari,"[drama, action, crime]",0.0,0.236433,False
2,Sara,maciejslesicki,"[drama, action, crime]",7.0,0.236433,False
3,Boca,fláviofrederico,"[action, crime, drama]",3.8,0.236433,False
4,Ghost Rider: Spirit of Vengeance,briantaylor,"[action, fantasy, thriller]",4.7,0.222277,False
5,Kickboxer 2: The Road Back,albertpyun,"[action, adventure]",4.2,0.221163,False
6,Romeo Is Bleeding,petermedak,"[action, crime, drama]",5.7,0.215353,False
7,Quest,tyronmontgomery,"[animation, drama, action]",6.7,0.208514,False
8,One Way Out,allana.goldstein,"[action, crime, drama]",4.0,0.197814,False
9,Gangster's Paradise: Jerusalema,ralphziman,"[drama, action, crime]",6.8,0.188608,False


In [117]:
recomendaciones_peso_3 = recommend_with_director_weight(
    title="The Dark Knight Rises",
    dataframe=movies_demo,
    soup_column="soup_director_3",
    n=10
)

print("Recomendaciones aumentando el peso del director:")

recomendaciones_peso_3

Recomendaciones aumentando el peso del director:


,title,director,genres,vote_average,similarity,same_director
0,Nick Fury: Agent of S.H.I.E.L.D.,rodhardy,"[sciencefiction, action]",3.4,0.146944,False
1,Ghost Rider: Spirit of Vengeance,briantaylor,"[action, fantasy, thriller]",4.7,0.146176,False
2,Bella Bettien,,"[crime, drama]",6.4,0.143223,False
3,Romeo Is Bleeding,petermedak,"[action, crime, drama]",5.7,0.133556,False
4,Bare Knuckles,,"[action, adventure, drama]",6.0,0.130744,False
5,Sara,maciejslesicki,"[drama, action, crime]",7.0,0.124035,False
6,Gangsters,giuseppevari,"[drama, action, crime]",0.0,0.124035,False
7,Boca,fláviofrederico,"[action, crime, drama]",3.8,0.124035,False
8,Kickboxer 2: The Road Back,albertpyun,"[action, adventure]",4.2,0.120096,False
9,Evil Behind You,,"[action, thriller]",0.0,0.113228,False


***

***

# Comparación de los modelos


## Recomendador simple

Ventajas:
- Es fácil y rápido.
- Ayuda a encontrar películas populares y bien calificadas.
- No necesita información personal del usuario.

Limitaciones:
- Entrega casi las mismas opciones a todas las personas.
- No considera géneros, actores ni preferencias individuales.


## Recomendador por sinopsis

Ventajas:
- Encuentra películas con tramas o temas parecidos.
- Utiliza técnicas básicas de NLP mediante TF-IDF.

Limitaciones:
- Puede relacionar películas únicamente porque comparten palabras.
- No considera reparto, director ni calidad general.


## Recomendador mejorado

Ventajas:
- Combina actores, director, géneros y palabras clave.
- Produce recomendaciones más cercanas al estilo de la película consultada.

Limitaciones:
- Sigue sin aprender directamente de las preferencias de cada usuario.
- Depende de que los metadatos estén completos y bien escritos.

## Recomendador híbrido

Ventajas:
- Combina similitud con popularidad.
- Reduce recomendaciones poco conocidas o con calificaciones bajas.

Limitaciones:
- La popularidad puede favorecer películas comerciales y reducir la diversidad.


# SUGERENCIAS implementadas

## 1. Introducción de un filtro de popularidad

Como mejora al recomendador basado en metadatos, se incorporó un filtro de popularidad utilizando la calificación ponderada de IMDb. En lugar de devolver únicamente las películas más similares, el sistema selecciona primero las 30 películas con mayor similitud y posteriormente calcula para cada una su calificación ponderada utilizando el número de votos y la valoración promedio.

Finalmente, las películas se ordenan utilizando esta puntuación y se devuelven las 10 recomendaciones mejor posicionadas. Esta estrategia permite mantener la similitud temática mientras se priorizan películas con mejor aceptación por parte de los usuarios, reduciendo la aparición de títulos poco conocidos o con escasas valoraciones.

---

## 2. Incorporación de otros miembros del equipo

Otra mejora consistió en ampliar la información utilizada para representar cada película mediante la incorporación de otros miembros relevantes del equipo creativo, específicamente guionistas y productores.

Estos profesionales también influyen en el estilo, narrativa y calidad de una producción cinematográfica, por lo que incluir sus nombres dentro de la representación textual (`soup`) permite capturar relaciones adicionales entre películas que comparten talento detrás de cámaras.

Como resultado, el recomendador dispone de una representación más rica del contenido de cada película, aumentando la capacidad para identificar similitudes que no dependen únicamente del reparto o del director.

---

## 3. Aumento del peso del director

Finalmente, se incrementó la importancia del director dentro de la representación textual utilizada por el recomendador. Para ello, el nombre del director se repitió varias veces dentro de la variable `soup`, aumentando su frecuencia antes del proceso de vectorización mediante `CountVectorizer`.

Esta modificación incrementa el peso estadístico del director durante el cálculo de similitud, favoreciendo que películas dirigidas por la misma persona obtengan puntuaciones de similitud más elevadas.

Durante las pruebas realizadas, en el presente ejercicio no se observó un incremento significativo en las recomendaciones de películas dirigidas por la misma persona. Este resultado puede atribuirse al sesgo introducido por el uso de una muestra aleatoria de películas para reducir el consumo de memoria (5,000 titulos en vez del total de 45,432), la cual no garantizaba una representación suficiente de la filmografía de cada director. En consecuencia, aunque la tendencia teórica de esta mejora es válida, los resultados obtenidos no permiten evidenciar completamente su impacto bajo las condiciones experimentales utilizadas (11%).

